# capture: square crops + automatic labels from a PGN replay (M2)

Data collection without hand labeling: replay a recorded game (PGN) on the physical board,
one position per ply. python-chess knows the true board at every ply, so each warped frame
is labeled automatically: 64 crops, 64 labels, 64 manifest rows. Output format is the
contract in `docs/contracts.md` (read back by `dataset.ipynb`):

```
data/crops/<session_id>/frames/<frame_id>.png
                        squares/<frame_id>_<square>.png
                        manifest.csv
```

Crop geometry (`configs/camera.yaml: crop`): the square, widened by `widen` squares on the
left and on the right, extended by `extend_up` squares toward the top of the warped image
(pieces are tall and lean away from the camera, i.e. upward in the `black_bottom` warp),
then resized to `size_wh`. Parts outside the warp are padded by edge replication.
Note: with the defaults (widen 0.5, extend_up 1.0) the region is 2 x 2 squares and is resized
to 100 x 200, i.e. squeezed 2x horizontally. That is the contract as written; revisit
(e.g. widen 0.0 -> 1 x 2 squares, no distortion) before collecting a lot of data.

Milestone: M2 (data collection). Run with `run_capture.ipynb`.

**TODO**
- [ ] crop geometry: `crop_box`, `extract_crop`, `extract_all_crops`
- [ ] auto-labeling: `pgn_positions`, `label_rows`
- [ ] `ManifestWriter` (frame ids, manifest append, write_frame)
- [ ] `capture_game`: live camera loop (prompt the human to set each position, grab, undistort,
      warp, crop, write)
- [ ] decide the final crop numbers (see the note above) and record them in camera.yaml
- [ ] simulated sessions: same writer fed by Gazebo frames (`gambit_sim` data generator)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

import csv
import io
from pathlib import Path

import chess
import chess.pgn
import numpy as np

%run ./homography.ipynb
%run ./dataset.ipynb

## Crop geometry

In [ ]:
def crop_box(sq: int, geom: BoardGeometry, warp, crop) -> tuple[int, int, int, int]:
    """(x0, y0, x1, y1) of square sq's crop in the warp image (may extend past the edges)."""
    # Design:
    # - Start from square_cell_warp; widen by crop.widen * width on left and right; extend
    #   the top edge up by crop.extend_up * height; keep the bottom edge (the square's base).
    # - Do not clip here: extract_crop pads instead, so every crop has the same geometry.
    # TODO: implement
    raise NotImplementedError


def extract_crop(warped: np.ndarray, sq: int, geom: BoardGeometry, warp, crop) -> np.ndarray:
    """One crop, resized to crop.size_wh; out-of-image parts are edge-replicated."""
    # Design:
    # - Pad the warp with cv2.copyMakeBorder(BORDER_REPLICATE) just enough to contain the box,
    #   shift the box by the pad, slice, cv2.resize to size_wh (INTER_AREA when shrinking).
    # TODO: implement
    raise NotImplementedError


def extract_all_crops(warped: np.ndarray, geom: BoardGeometry, warp, crop) -> list[np.ndarray]:
    """64 crops in square-index order (a1..h8)."""
    # TODO: implement
    raise NotImplementedError

## Auto-labeling from a PGN
`pgn_positions` yields (move_idx, board) for the start position and after every ply of the
mainline; `label_rows` turns one board into the 64 manifest rows.

In [ ]:
def pgn_positions(pgn: str | Path) -> list[tuple[int, chess.Board]]:
    """[(0, start), (1, after ply 1), ...] for the first game in a PGN file or PGN text."""
    # Design:
    # - Accept a path (Path or str ending in .pgn) or the PGN text itself.
    # - chess.pgn.read_game(io.StringIO(text)); None -> ValueError.
    # - Push game.mainline_moves() one by one onto game.board(), store a copy after each.
    # TODO: implement
    raise NotImplementedError


def label_rows(session_id: str, game_id: str, move_idx: int, frame_id: str,
               board: chess.Board) -> list[dict]:
    """64 manifest rows (MANIFEST_COLUMNS) for one frame, square-index order.

    path = "squares/<frame_id>_<square>.png" (relative to the session dir), fen = board.fen().
    """
    # TODO: implement (board_to_labels + SQUARE_NAMES)
    raise NotImplementedError

## Writing a session
`ManifestWriter` appends rows to `manifest.csv` (header written once) and hands out
frame ids (`000000`, `000001`, ... continuing after existing frames).

In [ ]:
class ManifestWriter:
    def __init__(self, crops_root: str | Path, session_id: str):
        # Design:
        # - Session dir = crops_root / session_id with squares/ and frames/ created.
        # - Continue numbering after the highest existing frames/*.png, so re-running a
        #   capture appends instead of overwriting.
        # TODO: implement
        raise NotImplementedError

    def next_frame_id(self) -> str:
        """Zero-padded 6-digit id, incremented on every call."""
        # TODO: implement
        raise NotImplementedError

    def append(self, rows: list[dict]) -> None:
        """Append rows to manifest.csv (csv.DictWriter), header only when the file is new."""
        # TODO: implement
        raise NotImplementedError

    def write_frame(self, warped: np.ndarray, crops: list[np.ndarray], game_id: str,
                    move_idx: int, board: chess.Board) -> str:
        """Save the warped frame, its 64 crops (BGR, as from cv2) and manifest rows; return frame id."""
        # Design: check 64 crops (ValueError), new frame id, write frames/<id>.png, label_rows,
        # write each crop to its row's path, append the rows.
        # Decision for you: write images first, manifest last, so a crash never leaves rows
        # pointing at missing files.
        # TODO: implement
        raise NotImplementedError

## Live capture loop

In [ ]:
def capture_game(camera, pgn: str | Path, writer: ManifestWriter, game_id: str,
                 geom: BoardGeometry, cam_cfg, intrinsics: dict | None = None,
                 frames_per_position: int = 1) -> int:
    """Replay one game on the physical board and record it. Returns the number of frames written."""
    # Design:
    # - For each (move_idx, board) in pgn_positions(pgn): print the move to make (or the
    #   full position), wait for the human (keypress).
    # - Grab frames_per_position frames (vary hand placement / lighting between them if you
    #   like), undistort with intrinsics (calibration.ipynb: undistort).
    # - compute_homography + warp_board, extract_all_crops, writer.write_frame(...).
    # - Skip frames where the markers are not found; show draw_grid_overlay as a live sanity check.
    # TODO: implement
    raise NotImplementedError

## Self-check
Crop geometry on the default config, crops from a synthetic warped board, PGN labeling and a
manifest round trip through `dataset.ipynb: load_manifest`.

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - Default config (camera.yaml warp + crop, black_bottom): square_cell_warp(a1) == (800, 100, 900, 200),
#   crop_box(a1) == (750, 0, 950, 200), crop_box(h8) == (50, 700, 250, 900).
# - extract_all_crops on a 1000x1000 synthetic warp (render_synthetic_board with square_px=100,
#   margin_px=150, trimmed by 50 px each side) -> 64 crops of shape (200, 100, 3).
# - pgn_positions('[Event "t"]\n\n1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. O-O *') -> 8 positions,
#   white king on g1 at the end.
# - label_rows for the position after 1. e4: 64 rows, e4 "P", e2 "empty",
#   row 0 path "squares/000001_a1.png".
# - ManifestWriter in a tempdir: 3 frames written -> load_manifest gives 192 rows,
#   frame ids 000000..000002, crop files exist; a new writer continues at "000003".
print("capture: self-check not written yet (see plan above)")